# ZEST — Kaggle GPU smoke-test notebook

Runs the full 5-stage ZEST pipeline end-to-end on a **tiny ESD subset** to produce at least one
emotion-converted `.wav`. It reuses the shipped `f0.pickle` + HuBERT-token manifests, patches every
hardcoded path at runtime, and fixes the `pickle5` / `torch.load` portability bugs.

## Before running
1. **Settings** → Accelerator = **GPU**, Internet = **ON** (both need a phone-verified Kaggle account — Settings → Phone).
2. **Add the ESD dataset** → right sidebar **+ Add Input** → search and add
   **`nguyenthanhlim/emotional-speech-dataset-esd`**
   (<https://www.kaggle.com/datasets/nguyenthanhlim/emotional-speech-dataset-esd>).
   It mounts at `/kaggle/input/emotional-speech-dataset-esd` (the default `ESD_WAV_DIR` in Cell 1).
   *Without a dataset attached, Cell 4 stops with `attached inputs : []` — that means nothing is mounted.*

## How Cell 1 gets the repo (automatic)
- **Online (default):** clones `REPO_URL` from GitHub — needs **Internet ON**.
- **Offline fallback:** upload the repo as a Kaggle Dataset and set `REPO_SRC` to its mount
  (e.g. `/kaggle/input/zest-repo`); Cell 1 copies from there instead. *(Later cells still download
  HuBERT / wav2vec2 / SpeechBrain from HuggingFace, so a full run needs Internet ON regardless.)*

Then run cells top-to-bottom. Cell 4 auto-scans all of `/kaggle/input` for the wavs, so `ESD_WAV_DIR`
only has to be roughly right as long as the ESD dataset is attached.


In [ ]:
# ============ Cell 1: setup, paths, env ============
import os, sys, subprocess, shutil, ast, json
from pathlib import Path

# ---- EDIT THESE TO MATCH YOUR SETUP ----
REPO_URL    = "https://github.com/vanshpatil16/zest"   # ONLINE: cloned at runtime (Internet ON; repo PUBLIC, or use a token URL)
REPO_SRC    = "/kaggle/input/zest-repo"   # OFFLINE: an uploaded Kaggle Dataset copy of the repo (used automatically if present)
ESD_WAV_DIR = "/kaggle/input/emotional-speech-dataset-esd"   # ESD root - Add Data: nguyenthanhlim/emotional-speech-dataset-esd
# (Cell 4 also auto-scans all of /kaggle/input and can kagglehub-download ESD, so this just needs to be roughly right.)
# -----------------------------------------

REPO = "/kaggle/working/ZEST"
CODE = REPO + "/code"
WORK = "/kaggle/working/zest"

# Get the repo into the writable working area.
# Prefer an uploaded dataset copy (works with Internet OFF); else clone from GitHub (needs Internet ON).
def _find_repo_root(src):
    """Dir under `src` that contains code/ - handles zip layouts with an extra wrapper folder."""
    if not src or not os.path.isdir(src):
        return None
    if os.path.isdir(os.path.join(src, "code")):
        return src
    for root, dirs, _ in os.walk(src):
        if "code" in dirs:
            return root
    return None

if not os.path.isdir(CODE):
    offline_root = _find_repo_root(REPO_SRC)
    if offline_root:
        print("Using OFFLINE repo copy from", offline_root)
        if os.path.isdir(REPO):
            shutil.rmtree(REPO)
        shutil.copytree(offline_root, REPO)
    else:
        print("Cloning repo from", REPO_URL, "(needs Internet ON)")
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
        except subprocess.CalledProcessError:
            msg = (
                "\n" + "=" * 70 +
                "\nREPO UNAVAILABLE - clone failed and no offline copy was found."
                "\nFix ONE of these, then re-run this cell:"
                "\n  (A) Notebook settings -> Internet = ON. Requires a phone-verified"
                "\n      Kaggle account (Settings -> Phone) - which also unlocks the GPU."
                "\n  (B) Upload the ZEST repo as a Kaggle Dataset, then set REPO_SRC to"
                "\n      its mount (e.g. /kaggle/input/zest-repo)."
                "\n      NOTE: later cells download HuBERT / wav2vec2 / SpeechBrain from"
                "\n      HuggingFace, so a FULL run still needs Internet ON regardless."
                "\n" + "=" * 70
            )
            raise RuntimeError(msg) from None

P = {
    "DATA":           WORK + "/data",
    "TRAIN_DIR":      WORK + "/data/train",
    "VAL_DIR":        WORK + "/data/val",
    "TEST_DIR":       WORK + "/data/test",
    "XVECTOR_DIR":    WORK + "/x_vectors",
    "EASE_EMB_DIR":   WORK + "/EASE_embeddings",
    "F0_CONTOUR_DIR": WORK + "/f0_contours",
    "WAV2VEC_DIR":    WORK + "/wav2vec_feats",
    "PRED_DSDT_DIR":  WORK + "/pred_DSDT_f0",
    "CKPT_DIR":       WORK + "/checkpoints",
    "OUTPUT_DIR":     WORK + "/converted",
}
for d in P.values():
    os.makedirs(d, exist_ok=True)

# subset manifests we will write in Cell 4 (audio paths rewritten to the copied wavs)
P["TRAIN_MANIFEST"] = WORK + "/train_subset.txt"
P["VAL_MANIFEST"]   = WORK + "/val_subset.txt"
P["TEST_MANIFEST"]  = WORK + "/test_subset.txt"
# shipped artifacts reused as-is
P["F0_PICKLE"]      = CODE + "/f0.pickle"
F0_STATS            = CODE + "/esd_f0_stats.pth"
HIFIGAN_CONFIG      = WORK + "/hifigan_kaggle.json"

# smoke-size knobs (raise these for a fuller run)
P["EASE_EPOCHS"]    = "3"
P["F0_EPOCHS"]      = "2"
P["F0_BATCH"]       = "2"        # F0-predictor batch size; lower to "1" if Cell 6 still hits CUDA OOM
P["HIFIGAN_STEPS"]  = "200"
UTTS_PER_BUCKET     = 2          # train wavs per (speaker, emotion)
VAL_UTTS            = 1          # val/test wavs per (speaker, emotion)

# environment passed to every child process
ENV = dict(os.environ)
ENV.update(P)
ENV["PYTHONPATH"] = CODE + os.pathsep + ENV.get("PYTHONPATH", "")
ENV["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduce CUDA fragmentation OOM
ENV["PYTHONUNBUFFERED"] = "1"   # stream child-process output live (no buffered 'silent' stages)

# wav2vec2 is loaded 3x in Cell 6. Unauthenticated HF calls get rate-limited and can stall for
# many minutes. Once the model is cached, load it offline so child stages make ZERO network calls.
import glob as _glob
_wav2vec_cached = _glob.glob(os.path.expanduser(
    "~/.cache/huggingface/hub/models--facebook--wav2vec2-large-robust-ft-swbd-300h*"))
if _wav2vec_cached:
    ENV["HF_HUB_OFFLINE"] = "1"
    ENV["TRANSFORMERS_OFFLINE"] = "1"
    print("wav2vec2 cached -> HF OFFLINE mode ON for child stages (instant load, no network stalls)")
else:
    print("wav2vec2 not cached yet -> the first F0 stage will download it from HF (needs Internet ON)")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - enable the accelerator!"))
print("repo ready (code/ present):", os.path.isdir(CODE), "| ESD_WAV_DIR exists:", os.path.isdir(ESD_WAV_DIR))

In [ ]:
# ============ Cell 2: install deps (drop pickle5 - it won't build on Kaggle's Python) ============
req = Path(REPO + "/requirements.txt")
kept = [l for l in req.read_text().splitlines() if "pickle5" not in l]
req.write_text("\n".join(kept) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
print("dependencies installed (pickle5 removed; stdlib pickle reads protocol 5)")

In [ ]:
# ============ Cell 3: patch hardcoded paths + portability bugs in the working copy ============
# Idempotent + re-runnable. Each replacement is (old, new) or (old, new, marker):
#   - if `marker` is given and already in the file -> skip (handles inserts where old is part of new)
#   - else if `old` present -> apply; else if 2-tuple and `new` already present -> skip; else error.
def patch(rel, repls):
    f = Path(CODE) / rel
    s = f.read_text()
    for repl in repls:
        old, new = repl[0], repl[1]
        marker = repl[2] if len(repl) > 2 else None
        if marker is not None and marker in s:
            continue                              # explicit marker => already applied
        if old in s:
            s = s.replace(old, new)
        elif marker is None and new in s:
            continue                              # patched form already present
        else:
            raise AssertionError("PATCH MISS in %s for:\n%s" % (rel, old))
    f.write_text(s)
    print("patched", rel)

patch("F0_predictor/config.py", [
    ('train_datasets = {"ESD":"/home/soumyad/emoconv/ESD/train"}',
     'import os\ntrain_datasets = {"ESD": os.environ["TRAIN_DIR"]}'),
    ('val_datasets = {"ESD":"/home/soumyad/emoconv/ESD/val"}',
     'val_datasets = {"ESD": os.environ["VAL_DIR"]}'),
    ('test_datasets = {"ESD":"/home/soumyad/emoconv/ESD/test"}',
     'test_datasets = {"ESD": os.environ["TEST_DIR"]}'),
    ('train_tokens_orig = {"ESD":"/ZEST/code/train_esd.txt"}',
     'train_tokens_orig = {"ESD": os.environ["TRAIN_MANIFEST"]}'),
    ('val_tokens_orig = {"ESD":"/ZEST/code/val_esd.txt"}',
     'val_tokens_orig = {"ESD": os.environ["VAL_MANIFEST"]}'),
    ('test_tokens_orig = {"ESD":"/ZEST/code/test_esd.txt"}',
     'test_tokens_orig = {"ESD": os.environ["TEST_MANIFEST"]}'),
    ('f0_file = "ZEST/code/f0.pickle"',
     'f0_file = os.environ["F0_PICKLE"]'),
])

patch("EASE/get_speaker_embedding.py", [
    ('folder = "/folder/to/wav_files"',          'folder = os.environ["EASE_WAV_DIR"]'),
    ('target_folder = "/folder/to/store/x-vectors"', 'target_folder = os.environ["XVECTOR_DIR"]'),
])

patch("EASE/speaker_classifier.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ('    speaker_folder = "/folder/to/x-vectors"', '    speaker_folder = os.environ["XVECTOR_DIR"]'),
    ('        folder = "/folder/to/train/audio/files"',      '        folder = os.environ["TRAIN_DIR"]'),
    ('        folder = "/folder/to/validation/audio/files"', '        folder = os.environ["VAL_DIR"]'),
    ('        folder = "/folder/to/test/audio/files"',       '        folder = os.environ["TEST_DIR"]'),
    ('    for e in range(10):', '    for e in range(int(os.environ.get("EASE_EPOCHS", "3"))):'),
    ("model = torch.load('EASE.pth', map_location=device)",
     "model = torch.load('EASE.pth', map_location=device, weights_only=False)"),
    ('os.makedirs("EASE_embeddings", exist_ok=True)', 'os.makedirs(os.environ["EASE_EMB_DIR"], exist_ok=True)'),
    ('np.save(os.path.join("EASE_embeddings", target_file_name)',
     'np.save(os.path.join(os.environ["EASE_EMB_DIR"], target_file_name)'),
])

patch("F0_predictor/pitch_attention_adv.py", [
    ('import pickle5 as pickle', 'import pickle'),
    # OOM fixes for the 14.5 GB T4: smaller batch + no anomaly graph retention
    ('torch.autograd.set_detect_anomaly(True)', 'torch.autograd.set_detect_anomaly(False)'),
    ('def create_dataset(mode, bs=24):', 'def create_dataset(mode, bs=int(os.environ.get("F0_BATCH", "2"))):'),
    ('np.load(os.path.join("/folder/to/EASE/embeddings", file_name.replace(".wav", ".npy")))',
     'np.load(os.path.join(os.environ["EASE_EMB_DIR"], file_name.replace(".wav", ".npy")))'),
    ('        folder = "/folder/to/train/audio/files"',      '        folder = os.environ["TRAIN_DIR"]'),
    ('        folder = "/folder/to/validation/audio/files"', '        folder = os.environ["VAL_DIR"]'),
    ('        folder = "/folder/to/test/audio/files"',       '        folder = os.environ["TEST_DIR"]'),
    ('    for e in range(500):', '    for e in range(int(os.environ.get("F0_EPOCHS", "2"))):'),
])

patch("F0_predictor/pitch_inference.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ("model.load_state_dict(torch.load('f0_predictor.pth', map_location=device))",
     "model.load_state_dict(torch.load('f0_predictor.pth', map_location=device, weights_only=False))"),
    ('os.makedirs("f0_contours", exist_ok=True)', 'os.makedirs(os.environ["F0_CONTOUR_DIR"], exist_ok=True)'),
    ('np.save(os.path.join("f0_contours", target_file_name)',
     'np.save(os.path.join(os.environ["F0_CONTOUR_DIR"], target_file_name)'),
])

patch("F0_predictor/get_wav2vec_feats.py", [
    ('from config import hparams, f0_stats', 'from config import hparams'),
    ('import pickle5 as pickle', 'import pickle'),
    ("model.load_state_dict(torch.load('f0_predictor.pth', map_location=device))",
     "model.load_state_dict(torch.load('f0_predictor.pth', map_location=device, weights_only=False))"),
    ('    wav2vec_feats_folder = "wav2vec_feats"', '    wav2vec_feats_folder = os.environ["WAV2VEC_DIR"]'),
])

patch("F0_predictor/pitch_convert.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ("model.load_state_dict(torch.load('f0_predictor.pth', map_location=device))",
     "model.load_state_dict(torch.load('f0_predictor.pth', map_location=device, weights_only=False))"),
    ('os.makedirs("pred_DSDT_f0", exist_ok=True)', 'os.makedirs(os.environ["PRED_DSDT_DIR"], exist_ok=True)'),
    ('np.save(os.path.join("pred_DSDT_f0", final_name)', 'np.save(os.path.join(os.environ["PRED_DSDT_DIR"], final_name)'),
    ('''    sources = ["0011_000021.wav", "0012_000022.wav", "0013_000025.wav",
               "0014_000032.wav", "0015_000034.wav", "0016_000035.wav",
               "0017_000038.wav", "0018_000043.wav", "0019_000023.wav",
               "0020_000047.wav"]''',
     '    sources = sorted(f for f in os.listdir(os.environ["TEST_DIR"]) if f.endswith(".wav") and int(f[5:11]) <= 350)'),
])

patch("HiFi-GAN/dataset.py", [
    ('import pickle5 as pickle', 'import pickle'),
    ('feats[\'spkr\'] = np.load("/ZEST/code/EASE/EASE_embeddings/" + emo_file_name)',
     'feats[\'spkr\'] = np.load(os.environ["EASE_EMB_DIR"] + "/" + emo_file_name)'),
    ("self.f0_stats = torch.load(f0_stats)",
     "self.f0_stats = torch.load(f0_stats, weights_only=False)"),
    ("mel = librosa_mel_fn(sampling_rate, n_fft, num_mels, fmin, fmax)",
     "mel = librosa_mel_fn(sr=sampling_rate, n_fft=n_fft, n_mels=num_mels, fmin=fmin, fmax=fmax)"),
])

patch("HiFi-GAN/inference.py", [
    ('reference_files = os.listdir("/folder/to/ESD/test/wavs")',
     'reference_files = os.listdir(os.environ["TEST_DIR"])'),
    ('emo_embed = np.load("/ZEST/code/F0_predictor/wav2vec_feats/" + filename.replace(".wav", ".npy"))',
     'emo_embed = np.load(os.environ["WAV2VEC_DIR"] + "/" + filename.replace(".wav", ".npy"))'),
    ('f0 = np.load("/ZEST/code/F0_predictor/pred_DSDT_f0" + fname_out_name + filename.replace(".wav", ".npy"))',
     'f0 = np.load(os.environ["PRED_DSDT_DIR"] + "/" + fname_out_name + filename.replace(".wav", ".npy"))'),
])

print("\nAll patches applied.")

In [ ]:
# ============ Cell 4: Stage 0 - build a tiny subset + rewrite manifest audio paths ============
# Reuses the shipped *_esd.txt (HuBERT tokens) and f0.pickle. Only the wav FILES come from ESD.
EMO_BOUNDS = [(0, 350), (351, 700), (701, 1050), (1051, 1400), (1401, 10**9)]
def bucket(fid):
    for i, (lo, hi) in enumerate(EMO_BOUNDS):
        if lo <= fid <= hi:
            return i
    return 4

def read_manifest(path):
    out = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line:
            out.append(ast.literal_eval(line))
    return out

def base(rec):
    return rec["audio"].split("/")[-1].split("\\")[-1]

# index every ESD wav by basename (first match wins). Three escalating sources:
#   1) ESD_WAV_DIR  2) a full scan of /kaggle/input  3) download via kagglehub (needs Internet ON)
def index_wavs(root_dir):
    idx = {}
    if root_dir and os.path.isdir(root_dir):
        for root, _, files in os.walk(root_dir):
            for fn in files:
                if fn.endswith(".wav"):
                    idx.setdefault(fn, os.path.join(root, fn))
    return idx

wav_index = index_wavs(ESD_WAV_DIR)
print("Indexed", len(wav_index), "wavs under", ESD_WAV_DIR)
if not wav_index:
    print("No wavs at ESD_WAV_DIR - scanning all of /kaggle/input ...")
    wav_index = index_wavs("/kaggle/input")
    print("Indexed", len(wav_index), "wavs under /kaggle/input")
if not wav_index:
    print("Nothing attached - trying to DOWNLOAD ESD via kagglehub (needs Internet ON; ~2 GB, cached after first run) ...")
    try:
        import kagglehub
        esd_path = kagglehub.dataset_download("nguyenthanhlim/emotional-speech-dataset-esd")
        print("kagglehub downloaded ESD to", esd_path)
        wav_index = index_wavs(esd_path)
        print("Indexed", len(wav_index), "wavs from the kagglehub download")
    except Exception as e:
        print("kagglehub download failed:", repr(e))

def build_split(full_manifest, dest_dir, manifest_out, per_bucket):
    recs = read_manifest(full_manifest)
    from collections import defaultdict
    grouped = defaultdict(list)
    for r in recs:
        bn = base(r)
        if bn in wav_index:                      # only keep wavs that actually exist in the ESD input
            grouped[(bn[:4], bucket(int(bn[5:11])))].append(r)
    written = 0
    lines = []
    for key in sorted(grouped):
        for r in sorted(grouped[key], key=base)[:per_bucket]:
            bn = base(r)
            dst = os.path.join(dest_dir, bn)
            if not os.path.exists(dst):
                shutil.copy2(wav_index[bn], dst)
            r2 = dict(r)
            r2["audio"] = dst.replace("\\", "/")  # rewrite to the copied location
            lines.append(str(r2))
            written += 1
    Path(manifest_out).write_text("\n".join(lines) + "\n")
    return written

n_tr = build_split(CODE + "/train_esd.txt", P["TRAIN_DIR"], P["TRAIN_MANIFEST"], UTTS_PER_BUCKET)
n_va = build_split(CODE + "/val_esd.txt",   P["VAL_DIR"],   P["VAL_MANIFEST"],   VAL_UTTS)
n_te = build_split(CODE + "/test_esd.txt",  P["TEST_DIR"],  P["TEST_MANIFEST"],  VAL_UTTS)
print(f"subset wavs -> train {n_tr}, val {n_va}, test {n_te}")
if min(n_tr, n_va, n_te) == 0:
    expected = [base(r) for r in read_manifest(CODE + "/train_esd.txt")[:5]]
    sample = list(wav_index)[:5]
    inputs = [str(p) for p in sorted(Path("/kaggle/input").glob("*"))] if os.path.isdir("/kaggle/input") else []
    raise RuntimeError(
        "\n" + "=" * 70 +
        "\nNO ESD WAVS MATCHED - a subset split is empty, so later stages would fail."
        f"\n  attached inputs : {inputs}"
        f"\n  wavs indexed    : {len(wav_index)}  sample: {sample}"
        f"\n  manifest expects: {expected}"
        "\nThe cell tried ESD_WAV_DIR, a /kaggle/input scan, AND a kagglehub download."
        "\nIf all three found nothing, EITHER Internet is OFF (kagglehub can't download),"
        "\nOR add the dataset manually: right sidebar -> + Add Input ->"
        "\n'nguyenthanhlim/emotional-speech-dataset-esd'. Then re-run this cell."
        "\n" + "=" * 70
    )

# write a Kaggle HiFi-GAN config from the shipped template
cfg = json.loads(Path(CODE + "/HiFi-GAN/hubert_alladv.json").read_text())
cfg["input_training_file"]   = P["TRAIN_MANIFEST"]
cfg["input_validation_file"] = P["VAL_MANIFEST"]
cfg["f0_stats"]   = F0_STATS
cfg["num_gpus"]   = 0
cfg["batch_size"] = 4
cfg["num_workers"] = 0
Path(HIFIGAN_CONFIG).write_text(json.dumps(cfg, indent=2))
print("HiFi-GAN config ->", HIFIGAN_CONFIG)

In [ ]:
# ============ Cell 5: Stage 1 - EASE (x-vectors -> adversarial speaker encoder -> embeddings) ============
import time, threading
EASE = CODE + "/EASE"

# Streaming runner used by Cells 5-8. Prints child output live, emits a heartbeat if a stage goes
# quiet (showing the LAST line it printed, i.e. where it is stuck), and reports elapsed time + exit
# code per stage. This turns opaque "is it hung?" stages into visible, diagnosable ones.
def run(cmd, cwd, env=None, idle_warn=90):
    cmd = [str(c) for c in cmd]
    print("\n>>", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, env=(env or ENV),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    start = time.time()
    last = [time.time(), ""]            # [time of last output, last line]
    stop = threading.Event()
    def heartbeat():
        while not stop.wait(idle_warn):
            idle = time.time() - last[0]
            if idle >= idle_warn:
                print(f"   ...alive: {int(time.time()-start)}s elapsed, {int(idle)}s since last line"
                      f" | last: {last[1][:90]!r}", flush=True)
    threading.Thread(target=heartbeat, daemon=True).start()
    for line in proc.stdout:
        last[0], last[1] = time.time(), line.rstrip()
        print(line, end="", flush=True)
    proc.wait(); stop.set()
    print(f"   [stage done in {int(time.time()-start)}s, exit {proc.returncode}]", flush=True)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

# x-vectors for all three splits (speaker_classifier reads them all from XVECTOR_DIR)
for folder in (P["TRAIN_DIR"], P["VAL_DIR"], P["TEST_DIR"]):
    e = dict(ENV); e["EASE_WAV_DIR"] = folder
    run([sys.executable, EASE + "/get_speaker_embedding.py"], EASE, e)
print("x-vectors:", len(list(Path(P["XVECTOR_DIR"]).glob("*.npy"))))

run([sys.executable, EASE + "/speaker_classifier.py"], EASE)
print("EASE embeddings:", len(list(Path(P["EASE_EMB_DIR"]).glob("*.npy"))))

In [ ]:
# ============ Cell 6: Stage 2 - F0 predictor (train -> contours -> SACE wav2vec feats) ============
F0 = CODE + "/F0_predictor"
run([sys.executable, F0 + "/pitch_attention_adv.py"], F0)   # writes f0_predictor.pth into F0/
run([sys.executable, F0 + "/pitch_inference.py"], F0)       # -> F0_CONTOUR_DIR
run([sys.executable, F0 + "/get_wav2vec_feats.py"], F0)     # -> WAV2VEC_DIR
print("f0_contours:", len(list(Path(P["F0_CONTOUR_DIR"]).glob("*.npy"))),
      "| wav2vec_feats:", len(list(Path(P["WAV2VEC_DIR"]).glob("*.npy"))))

In [ ]:
# ============ Cell 7: Stage 3 - train HiFi-GAN (smoke: a few hundred steps) ============
HG = CODE + "/HiFi-GAN"
run([sys.executable, HG + "/train.py",
     "--checkpoint_path", P["CKPT_DIR"],
     "--config", HIFIGAN_CONFIG,
     "--pitch_folder", P["F0_CONTOUR_DIR"] + "/",
     "--emo_folder",   P["WAV2VEC_DIR"] + "/",
     "--training_steps", P["HIFIGAN_STEPS"],
     "--checkpoint_interval", P["HIFIGAN_STEPS"]], HG)
print("checkpoints:", [p.name for p in Path(P["CKPT_DIR"]).glob("g_*")])

In [ ]:
# ============ Cell 8: Stage 4 - convert F0 (DSDT) + HiFi-GAN inference -> converted wav ============
F0 = CODE + "/F0_predictor"; HG = CODE + "/HiFi-GAN"
run([sys.executable, F0 + "/pitch_convert.py"], F0)
print("pred_DSDT_f0:", len(list(Path(P["PRED_DSDT_DIR"]).glob("*.npy"))))

run([sys.executable, HG + "/inference.py", "--convert",
     "--checkpoint_file", P["CKPT_DIR"],
     "--output_dir",      P["OUTPUT_DIR"],
     "--emo_folder",      P["WAV2VEC_DIR"] + "/",
     "--pitch_folder",    P["F0_CONTOUR_DIR"] + "/",
     "--f0-stats",        F0_STATS,
     "--input_code_file", P["TEST_MANIFEST"]], HG)

wavs = sorted(Path(P["OUTPUT_DIR"]).glob("*.wav"))
print("CONVERTED WAVS:", [w.name for w in wavs][:10])
import IPython.display as ipd
if wavs:
    ipd.display(ipd.Audio(str(wavs[0])))
else:
    print("No converted wavs. The test subset may lack a valid neutral-source / emotional-reference pair.")
    print("Fix: raise VAL_UTTS in Cell 1 (e.g. 3) so more speakers/emotions land in the test split, then re-run Cells 4 and 8.")

In [ ]:
# ============ Cell 9: run report ============
for label, d, pat in [("subset train", P["TRAIN_DIR"], "*.wav"), ("subset test", P["TEST_DIR"], "*.wav"),
                      ("x-vectors", P["XVECTOR_DIR"], "*.npy"), ("EASE emb", P["EASE_EMB_DIR"], "*.npy"),
                      ("f0_contours", P["F0_CONTOUR_DIR"], "*.npy"), ("wav2vec_feats", P["WAV2VEC_DIR"], "*.npy"),
                      ("pred_DSDT_f0", P["PRED_DSDT_DIR"], "*.npy"), ("checkpoints", P["CKPT_DIR"], "g_*"),
                      ("CONVERTED wavs", P["OUTPUT_DIR"], "*.wav")]:
    print(f"{label:16s}: {len(list(Path(d).glob(pat)))}")